# API 인증키 확인

`.env`의 인증키로 외부 API를 한 번씩 호출해 응답을 그대로 본다. 셀마다 독립적으로 실행할 수 있다.

- 출력에서는 인증키 값을 `***`로 가린다.
- `.env`를 고친 뒤에는 **1번 셀부터 다시 실행**해야 새 키가 반영된다(`dotenv_values`는 실행할 때 파일을 읽는다).
- 공공데이터포털 오류 코드: `20` 활용 미승인, `22` 일일 트래픽 초과, `30` 등록되지 않은 서비스키, `31` 기한 만료

In [2]:
import re
from pathlib import Path

import requests
from dotenv import dotenv_values

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists():
  ROOT = ROOT.parent

env = dotenv_values(ROOT / ".env")
SECRETS = [v for v in env.values() if v and len(v) >= 4]


def mask(text: str) -> str:
  for v in SECRETS:
    text = text.replace(v, "***")
  return text


def show(resp: requests.Response, limit: int = 1500) -> None:
  print("HTTP", resp.status_code, "|", resp.headers.get("content-type"))
  print(mask(resp.text[:limit]))
  

for name in ["DATA_GO_KR_KEY", "YOUTHCENTER_KEY", "OC", "QDRANT_URL", "QDRANT_API_KEY"]:
  v = env.get(name) or ""
  print(f"{name:16} {'있음' if v else '없음':3} 길이 {len(v):3}  {v[:4] + '…' + v[-4:] if len(v) > 8 else ''}")

DATA_GO_KR_KEY   있음  길이  64  55f6…5d7c
YOUTHCENTER_KEY  있음  길이  36  f188…f612
OC               있음  길이   4  
QDRANT_URL       있음  길이  76  http…t.io
QDRANT_API_KEY   있음  길이 176  eyJh…TfL8


## 1. 생활법령 백문백답 (SOAP, `DATA_GO_KR_KEY`)

사용자 가이드 v4.0의 요청 예제 그대로다. 인증키는 SOAP 헤더 `ComMsgHeader/ServiceKey`에 들어간다.
가이드 주소는 `http://`지만 `https://`로 리다이렉트되므로 `https`로 보낸다.

In [3]:
EASYLAW_URL = "https://www.easylaw.go.kr/OPENAPI/soap/ManyAskManyAnswerService"

envelope = f"""<soapenv:Envelope xmlns:soapenv="http://schemas.xmlsoap.org/soap/envelope/" xmlns:head="http://apache.org/headers" xmlns:open="http://openapi.affis.go.kr">
   <soapenv:Header>
      <head:ComMsgHeader>
         <RequestMsgID>?</RequestMsgID>
         <ServiceKey>{env["DATA_GO_KR_KEY"]}</ServiceKey>
      </head:ComMsgHeader>
   </soapenv:Header>
   <soapenv:Body>
      <open:getMaskMAnswerClassList/>
   </soapenv:Body>
</soapenv:Envelope>"""


print("보내는 요청:\n" + mask(envelope) + "\n")
resp = requests.post(EASYLAW_URL, data=envelope.encode("utf-8"), headers={"Content-Type": "text/xml; charset=utf-8"}, timeout=60)
print("returnCode / errMsg:", re.findall(r"<(returnCode|errMsg)>([^<]*)<", resp.text) or "없음(정상)")
show(resp)

보내는 요청:
<soapenv:Envelope xmlns:soapenv="http://schemas.xmlsoap.org/soap/envelope/" xmlns:head="http://apache.org/headers" xmlns:open="http://openapi.affis.go.kr">
   <soapenv:Header>
      <head:ComMsgHeader>
         <RequestMsgID>?</RequestMsgID>
         <ServiceKey>***</ServiceKey>
      </head:ComMsgHeader>
   </soapenv:Header>
   <soapenv:Body>
      <open:getMaskMAnswerClassList/>
   </soapenv:Body>
</soapenv:Envelope>

returnCode / errMsg: [('errMsg', 'SERVICE KEY IS NOT REGISTERED ERROR.'), ('returnCode', '30')]
HTTP 200 | text/xml;charset=UTF-8
<soap:Envelope xmlns:soap="http://schemas.xmlsoap.org/soap/envelope/"><soap:Header><ns3:ComMsgHeader xmlns:ns3="http://apache.org/headers" xmlns:ns2="http://openapi.affis.go.kr"><RequestMsgID>?</RequestMsgID><ServiceKey>***</ServiceKey></ns3:ComMsgHeader></soap:Header><soap:Body><ns2:getMaskMAnswerClassListResponse xmlns:ns2="http://openapi.affis.go.kr"><MaskMAnswerClassListResponse><errMsg>SERVICE KEY IS NOT REGISTERED ERROR.</errMsg

## 2. 복지로 중앙부처 복지서비스 (REST, `DATA_GO_KR_KEY`)

같은 키로 다른 API를 불러 **키 자체가 유효한지** 대조한다. 여기서 정상이면 키는 맞고, 1번 실패는 해당 서비스 쪽 등록 문제다.

In [4]:
resp = requests.get(
  "https://apis.data.go.kr/B554287/NationalWelfareInformationsV001/NationalWelfarelistV001",
  params={"serviceKey": env["DATA_GO_KR_KEY"], "callTp": "L", "pageNo": 1, "numOfRows": 3, "srchKeyCode": "001", "searchWrd": "주거"},
  timeout=60,
)
show(resp)

HTTP 200 | application/xml;charset=utf-8
<?xml version="1.0" encoding="UTF-8" standalone="yes"?><wantedList><totalCount>7</totalCount><pageNo>1</pageNo><numOfRows>3</numOfRows><resultCode>0</resultCode><resultMessage>SUCCESS</resultMessage><servList><inqNum>49809</inqNum><intrsThemaArray>생활지원,주거</intrsThemaArray><jurMnofNm>국토교통부</jurMnofNm><jurOrgNm>주거복지정책과</jurOrgNm><onapPsbltYn>N</onapPsbltYn><rprsCtadr>1599-0001</rprsCtadr><servDgst>쪽방·고시원 등 열악한 비주택거주자의 공공임대주택 이주수요를 발굴하고 LH와 협력하여 이주과정을 지원합니다.</servDgst><servDtlLink>https://www.bokjiro.go.kr/ssis-tbu/twataa/wlfareInfo/moveTWAT52011M.do?wlfareInfoId=WLF00000036&amp;wlfareInfoReldBztpCd=01</servDtlLink><servId>WLF00000036</servId><servNm>주거취약계층 주거상향 지원사업</servNm><sprtCycNm>1회성</sprtCycNm><srvPvsnNm>기타</srvPvsnNm><svcfrstRegTs>20210903</svcfrstRegTs><trgterIndvdlArray>저소득</trgterIndvdlArray></servList><servList><inqNum>88154</inqNum><intrsThemaArray>주거</intrsThemaArray><jurMnofNm>보건복지부</jurMnofNm><jurOrgNm>기초생활보장과</jurOrgNm><onapPsbltYn

## 3. 온통청년 청년정책 (`YOUTHCENTER_KEY`)

In [5]:
resp = requests.get(
  "https://www.youthcenter.go.kr/go/ythip/getPlcy",
  params={"apiKeyNm": env["YOUTHCENTER_KEY"], "pageNum": 1, "pageSize": 2, "rtnType": "json", "plcyNm": "월세"},
  timeout=60,
)
show(resp, limit=800)

HTTP 200 | application/json
{"resultCode":200,"resultMessage":"성공적으로 데이터를 가지고 왔습니다.","result":{"pagging":{"totCount":63,"pageNum":1,"pageSize":2},"youthPolicyList":[{"plcyNo":"20261002005400213823","bscPlanCycl":"2","bscPlanPlcyWayNo":"003","bscPlanFcsAsmtNo":"009","bscPlanAsmtNo":"030","pvsnInstGroupCd":"0054002","plcyPvsnMthdCd":"0042006","plcyAprvSttsCd":"0044002","plcyNm":"청년월세 지원","plcyKywdNm":"주거지원,보조금","plcyExplnCn":"경제적 어려움을 겪고 있는 저소득층 청년의 주거비 부담 경감을 위해 월세를 지원하여 주거 안정 도모","lclsfNm":"주거","mclsfNm":"전월세 및 주거급여 지원","plcySprtCn":" ㅇ (사업 대상) 부모와 별도 거주하는 무주택 청년\n   - (연령 기준) 19~34세\n   - (소득 기준) 기준중위소득 청년가구 60% 이하, 원가구 100% 이하\n   - (기타 조건) 재산가액 청년 가구122백만원, 원가구 470백만원 이하","sprvsnInstCd":"5690000","sprvsnInstCdNm":"세종특별자치시","sprvsnInstPicNm":"","operInstCd":"       ","operInstCdNm":"","operInstPicNm":"","sprtSclLmt


## 4. 법령 API (`OC`)

In [6]:
resp = requests.get(
  "https://www.law.go.kr/DRF/lawSearch.do",
  params={"OC": env["OC"], "target": "law", "type": "JSON", "query": "주택임대차보호법", "display": 1},
  timeout=60,
)
show(resp, limit=800)

HTTP 200 | application/json;charset=UTF-8
{"LawSearch": {
    "law": {
        "현행연혁코드": "현행",
        "법령일련번호": "276291",
        "자법타법여부": "",
        "법령상세링크": "/DRF/lawService.do?OC=***&target=law&MST=276291&type=HTML&mobileYn=&efYd=20260102",
        "법령명한글": "주택임대차보호법",
        "법령구분명": "법률",
        "소관부처명": "국토교통부,법무부",
        "공포번호": "21065",
        "제개정구분명": "타법개정",
        "소관부처코드": "1270000,1613000",
        "id": "1",
        "법령ID": "001248",
        "공동부령정보": "",
        "시행일자": "20260102",
        "공포일자": "20251001",
        "법령약칭명": "주택임대차법"
    },
    "resultMsg": "success",
    "키워드": "주택임대차보호법",
    "page": "1",
    "resultCode": "00",
    "target": "law",
    "totalCnt": "2",
    "section": "lawNm",
    "numOfRows": "1"
}}


## 5. Qdrant (`QDRANT_URL`, `QDRANT_API_KEY`)

In [7]:
resp = requests.get(env["QDRANT_URL"].rstrip("/") + "/collections", headers={"api-key": env["QDRANT_API_KEY"]}, timeout=30)
show(resp)

HTTP 200 | application/json
{"result":{"collections":[]},"status":"ok","time":4.956e-6}
